# Part 1: unconstrained static pricing

Run `model_training/purchase_model_baseline.ipynb` first. This example loads its pickled model and optimizes prices for a validation set and the true test set with the three required Part 1 columns. Replace this simple baseline with your own model and strategy.


In [ ]:
from pathlib import Path
import os
import sys

# Resolve paths once so every notebook uses the project folder.
project_folder = next((
    folder for parent in (Path.cwd(), *Path.cwd().parents)
    for folder in (parent, parent / 'project_code_2026_publicshare',
                   parent / 'project_2026/project_code_2026_publicshare',
                   parent / 'private_github/project_2026/project_code_2026_publicshare')
    if (folder / 'pricing_helpers.py').is_file()
), None)
if project_folder is None:
    raise FileNotFoundError('Open the project folder as your notebook workspace.')
os.chdir(project_folder)
sys.path.insert(0, str(project_folder))

import pickle
import numpy as np
import pandas as pd
from pricing_helpers import STATIC_PRICE_FILENAME, evaluate_static_prices

with (project_folder / 'agents/helper_files_here/purchase_model.pkl').open('rb') as file:
    model = pickle.load(file)  # Only load pickle files you trust.
    
training = pd.read_csv(project_folder / 'data/train_prices_decisions_2026.csv')
validation_customers = pd.read_csv(project_folder / 'data/validation_user_info_2026.csv')
test_customers = pd.read_csv(project_folder / 'data/test_user_info_2026.csv')
covariates = ['Covariate1', 'Covariate2', 'Covariate3']

## A naive pricing strategy

Choose each customer's price independently to maximize price times predicted purchase probability on a small grid.


In [ ]:
price_grid = np.linspace(0, 500, 251)  # Fine grid for pricing optimization


In [ ]:
### REVENUE-MAXIMIZING PRICING STRATEGY

def predicted_revenue(customers, prices):
    features = customers[covariates].assign(price_item=prices)
    # Clip regression predictions; for a classifier, predict_proba can be used
    probability = np.clip(model.predict(features), 0, 1)
    return np.asarray(prices) * probability


def choose_static_prices(customers):
    n = len(customers)
    personalized_prices = np.zeros(n)
    personalized_revenue = np.zeros(n)
    chunk_size = 2000
    for i in range(0, n, chunk_size):
        chunk = customers.iloc[i:i + chunk_size]
        chunk_len = len(chunk)
        repeated_covs = np.repeat(chunk[covariates].values, len(price_grid), axis=0)
        tiled_prices = np.tile(price_grid, chunk_len)
        features = pd.DataFrame(repeated_covs, columns=covariates)
        features["price_item"] = tiled_prices
        probs = np.clip(model.predict(features), 0, 1)
        revs = (tiled_prices * probs).reshape((chunk_len, len(price_grid)))
        best_idx = np.argmax(revs, axis=1)
        personalized_prices[i:i + chunk_len] = price_grid[best_idx]
        personalized_revenue[i:i + chunk_len] = revs[np.arange(chunk_len), best_idx]
    return personalized_prices, personalized_revenue


In [ ]:
def make_static_price_file(customers, filename, overwrite=True):
    filename = Path(filename)
    if filename.exists() and not overwrite:
        raise FileExistsError(f'{filename} already exists; keep it or explicitly allow replacement.')
    submission = customers[['user_index']].copy()
    prices, revenue = choose_static_prices(customers)
    submission['price_item'], submission['expected_revenue'] = prices, revenue
    submission.to_csv(filename, index=False)
    print('Wrote:', filename, '| Customers:', len(submission))
    return pd.DataFrame([{'customers': len(customers),
                          'predicted_average_revenue': revenue.mean()}])


## Validation: calculate realized performance offline

Write prices for `validation_user_info_2026.csv`, then call the evaluator. It uses the supplied encrypted validation valuations internally and returns aggregate results, not the labels. Compare predicted and realized average revenue; repeated tuning can overfit this validation set.

The validation CSV is for local testing only, not the required submission. Rerunning this notebook replaces its generated CSVs by default. Set `overwrite_csvs = False` to protect files from replacement.


In [ ]:
overwrite_csvs = True
validation_file = project_folder / 'model_training/static_prices_validation.csv'
print('Predicted validation performance:')
make_static_price_file(validation_customers, validation_file, overwrite=overwrite_csvs)


In [ ]:
print('Realized validation performance:')
evaluate_static_prices(validation_file, project_folder / 'data')


## Write the true test submission

Now apply your strategy to `test_user_info_2026.csv`. Its customer IDs differ from the validation IDs. Write the required file to `agents/static_prices_submission.csv`; the upload check rejects validation customers in its place. Test valuations are not supplied, so this table shows only **predicted** test performance. Staff score realized test revenue separately.


In [ ]:
test_file = project_folder / 'agents' / STATIC_PRICE_FILENAME
print('Predicted test performance:')
make_static_price_file(test_customers, test_file, overwrite=overwrite_csvs)
